# Chapter 3: Parallelization

EliteAI Academy

Run independent tasks with asyncio, threads, and LCEL; combine results, benchmark execution, and handle partial failures.

## 0. Setup

Run `setup.ps1` from the course folder once, then select **Python (Agentic AI Course)** as the kernel.
The course loads your DeepSeek credentials from `.env`; restart the kernel after editing that file.
The install cell below is optional if setup is already complete. Run the remaining cells in order.

**Teaching check:** explain the pattern, run one example, inspect its output, then try an exercise.
Live model calls use API credits and may produce different answers. Stock prices, weather, CRM records,
email delivery, and approval workflows in these lessons are classroom simulations. Model-generated
research is illustrative and needs independent source verification before factual use.


In [ ]:
%pip install -r requirements-lock.txt --quiet

In [ ]:
from course_config import MODEL
import os, json, time, asyncio
import concurrent.futures
from typing import List, Dict, Any

PROVIDER = "deepseek"

if PROVIDER == "deepseek":
    import anthropic
    _client = anthropic.Anthropic(timeout=60.0, max_retries=2)
    _model  = MODEL

    def call_llm(prompt: str, system: str = "You are a helpful AI assistant.") -> str:
        r = _client.messages.create(thinking={"type": "disabled"}, 
            model=_model, max_tokens=512, system=system,
            messages=[{"role": "user", "content": prompt}]
        )
        return r.content[0].text.strip()

    async def acall_llm(prompt: str, system: str = "You are a helpful AI assistant.") -> str:
        'Async version — uses a thread pool so the event loop stays free.'
        loop = asyncio.get_event_loop()
        return await loop.run_in_executor(None, lambda: call_llm(prompt, system))

elif PROVIDER == "gemini":
    import google.generativeai as genai
    genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
    _gmodel = genai.GenerativeModel("gemini-2.0-flash")

    def call_llm(prompt: str, system: str = "You are a helpful AI assistant.") -> str:
        return _gmodel.generate_content(f"{system}\n\n{prompt}").text.strip()

    async def acall_llm(prompt: str, system: str = "You are a helpful AI assistant.") -> str:
        loop = asyncio.get_event_loop()
        return await loop.run_in_executor(None, lambda: call_llm(prompt, system))

print(f"✅ Provider: {PROVIDER}")
print(call_llm("Reply with exactly: PARALLEL READY"))

## 1. The Problem — Sequential Execution Is Slow

When tasks are **independent** of each other, running them one-by-one wastes time. Every API call blocks execution while the network responds.

In [ ]:
TOPICS = [
    "The history of the internet",
    "How photosynthesis works",
    "The causes of World War I",
]

def summarise(topic: str) -> str:
    return call_llm(f"Summarise in exactly 2 sentences: {topic}")

print("=== SEQUENTIAL EXECUTION ===")
t0 = time.perf_counter()

sequential_results = []
for topic in TOPICS:
    result = summarise(topic)
    sequential_results.append(result)
    print(f"  ✅ Done: {topic[:35]}...")

seq_time = time.perf_counter() - t0
print(f"\n⏱️  Sequential total time: {seq_time:.2f}s")
print("   (Each task waited for the previous one to finish)")

## 2. Solution A — `asyncio.gather` (Async Concurrency)

A single event loop overlaps waiting API calls. This is concurrency; speedup depends on the workload.

In [ ]:
async def async_summarise(topic: str) -> dict:
    result = await acall_llm(f"Summarise in exactly 2 sentences: {topic}")
    return {"topic": topic, "summary": result}

async def run_all_async(topics: list) -> list:
    'Run all summarise tasks concurrently with asyncio.gather.'
    tasks = [async_summarise(t) for t in topics]
    return await asyncio.gather(*tasks)

print("=== ASYNC PARALLEL EXECUTION (asyncio.gather) ===")
t0 = time.perf_counter()

async_results = await run_all_async(TOPICS)

async_time = time.perf_counter() - t0
print(f"⏱️  Async total time: {async_time:.2f}s")
print(f"🚀 Speedup vs sequential: {seq_time/async_time:.1f}x faster")

for r in async_results:
    print(f"\n📌 {r['topic']}\n   {r['summary'][:120]}...")

## 3. Solution B — `ThreadPoolExecutor` (Thread-Based Parallelism)

Runs each task in a **separate OS thread**. Better for synchronous APIs where you can't use `async/await`.

In [ ]:
def thread_summarise(topic: str) -> dict:
    result = call_llm(f"Summarise in exactly 2 sentences: {topic}")
    return {"topic": topic, "summary": result}

print("=== THREAD POOL PARALLEL EXECUTION ===")
t0 = time.perf_counter()

with concurrent.futures.ThreadPoolExecutor(max_workers=len(TOPICS)) as executor:
    future_to_topic = {executor.submit(thread_summarise, t): t for t in TOPICS}
    thread_results = []
    for future in concurrent.futures.as_completed(future_to_topic):
        result = future.result()
        thread_results.append(result)
        print(f"  ✅ Completed: {result['topic'][:40]}")

thread_time = time.perf_counter() - t0
print(f"\n⏱️  ThreadPool total time: {thread_time:.2f}s")
print(f"🚀 Speedup vs sequential: {seq_time/thread_time:.1f}x faster")

## 4. Concurrency vs Parallelism — Clarification

In [ ]:
explanation = """
╔══════════════════════╦═══════════════════════════════╦══════════════════════════════╗
║ Approach             ║ How It Works                  ║ Best For                     ║
╠══════════════════════╬═══════════════════════════════╬══════════════════════════════╣
║ Sequential           ║ One task at a time            ║ Dependent tasks, debugging   ║
╠══════════════════════╬═══════════════════════════════╬══════════════════════════════╣
║ asyncio.gather       ║ Single thread, cooperative    ║ I/O-bound tasks (API calls)  ║
║                      ║ switching on await points     ║ Most LLM agentic workflows   ║
╠══════════════════════╬═══════════════════════════════╬══════════════════════════════╣
║ ThreadPoolExecutor   ║ Multiple OS threads           ║ Sync APIs, mixed I/O tasks   ║
║                      ║ (GIL limits CPU parallelism)  ║                              ║
╠══════════════════════╬═══════════════════════════════╬══════════════════════════════╣
║ ProcessPoolExecutor  ║ Multiple OS processes         ║ CPU-bound tasks              ║
║                      ║ (bypasses GIL)                ║ e.g. heavy data processing   ║
╚══════════════════════╩═══════════════════════════════╩══════════════════════════════╝

Key insight: For LLM API calls, asyncio.gather and ThreadPoolExecutor both
achieve similar speedups because the bottleneck is NETWORK I/O, not CPU.
"""
print(explanation)

## 5. The Fan-Out → Synthesise Pattern

One input → Independent tasks → Combined output.

In [ ]:
TOPIC = "The history of space exploration"

async def task_summarise(topic: str) -> str:
    return await acall_llm(
        f"Summarise this topic concisely in 3 bullet points: {topic}",
        system="You are a concise summariser."
    )

async def task_questions(topic: str) -> str:
    return await acall_llm(
        f"Generate 3 thought-provoking questions about: {topic}",
        system="You are a curious researcher."
    )

async def task_key_terms(topic: str) -> str:
    return await acall_llm(
        f"List 7 key terms related to: {topic}. Comma-separated only.",
        system="You are a knowledge graph builder."
    )

async def fan_out_synthesise(topic: str) -> dict:
    print(f"🚀 Fanning out 3 parallel tasks for: '{topic}'")
    t0 = time.perf_counter()

    summary, questions, key_terms = await asyncio.gather(
        task_summarise(topic),
        task_questions(topic),
        task_key_terms(topic),
    )

    parallel_time = time.perf_counter() - t0
    print(f"✅ All 3 parallel tasks completed in {parallel_time:.2f}s")

    synthesis_prompt = f"""
    You have been given research on the topic: '{topic}'.
    
    Summary:
    {summary}
    
    Key Questions:
    {questions}
    
    Key Terms: {key_terms}
    
    Synthesise these into a comprehensive, well-structured 4-5 sentence overview.
    """
    final = await acall_llm(synthesis_prompt, system="You are an expert synthesiser.")

    return {
        "topic":      topic,
        "summary":    summary,
        "questions":  questions,
        "key_terms":  key_terms,
        "synthesis":  final,
        "time_taken": parallel_time,
    }

result = await fan_out_synthesise(TOPIC)

print("\n── PARALLEL OUTPUTS ──")
print(f"\n📝 Summary:\n{result['summary']}")
print(f"\n❓ Questions:\n{result['questions']}")
print(f"\n🔑 Key Terms:\n{result['key_terms']}")
print(f"\n🔗 Final Synthesis:\n{result['synthesis']}")

## 6. LangChain LCEL — `RunnableParallel`

The book's code example: three chains running side-by-side via `RunnableParallel`.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

lc_llm = ChatAnthropic(model_kwargs={"thinking": {"type": "disabled"}}, model=MODEL, temperature=0.7)

summarise_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Summarise the following topic concisely in 3 bullet points:"),
        ("user",   "{topic}")
    ]) | lc_llm | StrOutputParser()
)

questions_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Generate 3 thought-provoking questions about:"),
        ("user",   "{topic}")
    ]) | lc_llm | StrOutputParser()
)

terms_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "List 7 key terms related to the topic. Comma-separated only:"),
        ("user",   "{topic}")
    ]) | lc_llm | StrOutputParser()
)

map_chain = RunnableParallel({
    "summary":   summarise_chain,
    "questions": questions_chain,
    "key_terms": terms_chain,
    "topic":     RunnablePassthrough(),
})

synthesis_prompt = ChatPromptTemplate.from_messages([
    ("system", """Synthesise the following research into a comprehensive 4-5 sentence overview.

Summary: {summary}
Questions: {questions}
Key Terms: {key_terms}"""),
    ("user", "Topic: {topic}")
])

full_parallel_chain = map_chain | synthesis_prompt | lc_llm | StrOutputParser()

print("=== LangChain RunnableParallel ===")
t0 = time.perf_counter()
lc_result = full_parallel_chain.invoke("Quantum computing")
elapsed = time.perf_counter() - t0

print(f"⏱️  Completed in {elapsed:.2f}s")
print(f"\n🔗 Final Synthesis:\n{lc_result}")

## 7. Use Case 1 — Multi-Source Research Agent

Simulates querying multiple data sources in parallel: news, stock data, social media, company DB.

In [ ]:
import random

async def fetch_news(company: str) -> str:
    await asyncio.sleep(random.uniform(0.1, 0.3))
    return await acall_llm(f"Write a 1-sentence fake news headline about {company}.")

async def fetch_stock_data(company: str) -> str:
    await asyncio.sleep(random.uniform(0.1, 0.2))
    return f"[STOCK API] {company}: ${'%.2f' % (random.uniform(100,500))} (+{random.uniform(0,5):.1f}%)  (simulated)"

async def fetch_social_mentions(company: str) -> str:
    await asyncio.sleep(random.uniform(0.1, 0.25))
    return await acall_llm(f"Describe in 1 sentence the social media sentiment about {company}.")

async def fetch_company_profile(company: str) -> str:
    await asyncio.sleep(random.uniform(0.05, 0.15))
    return await acall_llm(f"Describe {company} in exactly 1 sentence (industry, founded, HQ).")

async def research_company(company: str) -> dict:
    print(f"🔎 Researching '{company}' across 4 sources in parallel...")
    t0 = time.perf_counter()

    news, stock, social, profile = await asyncio.gather(
        fetch_news(company),
        fetch_stock_data(company),
        fetch_social_mentions(company),
        fetch_company_profile(company),
    )

    elapsed = time.perf_counter() - t0
    print(f"✅ All 4 sources retrieved in {elapsed:.2f}s")

    report = await acall_llm(
        f"""Write a 3-sentence analyst brief for {company} using ONLY this data:
Profile: {profile}
Stock: {stock}
News: {news}
Social: {social}"""
    )

    return {"company": company, "news": news, "stock": stock,
            "social": social, "profile": profile, "report": report}

company_result = await research_company("Tesla")
print(f"\n📊 Analyst Brief:\n{company_result['report']}")

## 8. Use Case 2 — Customer Feedback Analysis

Run sentiment analysis, keyword extraction, category classification, and urgency detection simultaneously on each feedback item.

In [ ]:
FEEDBACK_ITEMS = [
    "The product stopped working after 2 days. I need a refund immediately!",
    "Absolutely love the new design. Very intuitive and fast.",
    "Delivery took 3 weeks. Not happy at all.",
]

async def analyse_single_feedback(text: str) -> dict:
    'Run 4 analysis tasks in parallel on a single feedback item.'
    sentiment_task  = acall_llm(f"Classify sentiment as POSITIVE, NEGATIVE, or NEUTRAL only: {text}")
    keywords_task   = acall_llm(f"Extract 3 keywords comma-separated: {text}")
    category_task   = acall_llm(f"Classify into one category (product/delivery/support/other): {text}")
    urgency_task    = acall_llm(f"Is this urgent? Reply YES or NO only: {text}")

    sentiment, keywords, category, urgent = await asyncio.gather(
        sentiment_task, keywords_task, category_task, urgency_task
    )
    return {
        "text":      text[:60] + "...",
        "sentiment": sentiment.strip(),
        "keywords":  keywords.strip(),
        "category":  category.strip(),
        "urgent":    urgent.strip(),
    }

async def analyse_all_feedback(items: list) -> list:
    'Analyse all feedback items in parallel (outer) with inner parallelism per item.'
    return await asyncio.gather(*[analyse_single_feedback(item) for item in items])

print("=== PARALLEL CUSTOMER FEEDBACK ANALYSIS ===")
print("Running 4 analysis tasks × 3 feedback items = 12 LLM calls in parallel\n")

t0 = time.perf_counter()
feedback_results = await analyse_all_feedback(FEEDBACK_ITEMS)
elapsed = time.perf_counter() - t0

print(f"⏱️  Total time: {elapsed:.2f}s for 12 parallel LLM calls\n")
for r in feedback_results:
    print(f"📩 '{r['text']}'")
    print(f"   Sentiment: {r['sentiment']} | Category: {r['category']} | Urgent: {r['urgent']}")
    print(f"   Keywords : {r['keywords']}\n")

## 9. Use Case 3 — Travel Planning Agent

Check flights, hotels, local events, and restaurants simultaneously.

In [ ]:
async def travel_plan(destination: str, dates: str, budget: str) -> dict:
    context = f"destination={destination}, dates={dates}, budget={budget}"

    flights_task  = acall_llm(f"Suggest realistic flight options (airline, price range) for: {context}")
    hotels_task   = acall_llm(f"Recommend 2 hotels with price range for: {context}")
    events_task   = acall_llm(f"List 3 local events or must-see attractions for: {context}")
    restaurants_task = acall_llm(f"Recommend 2 restaurants with cuisine type for: {context}")

    print(f"✈️  Planning trip to {destination} — fetching 4 sources in parallel...")
    t0 = time.perf_counter()

    flights, hotels, events, restaurants = await asyncio.gather(
        flights_task, hotels_task, events_task, restaurants_task
    )

    elapsed = time.perf_counter() - t0
    print(f"✅ All data gathered in {elapsed:.2f}s")

    summary = await acall_llm(
        f"""Write a friendly 5-sentence travel plan summary for {destination}.
        
Flights: {flights}
Hotels: {hotels}
Events: {events}
Restaurants: {restaurants}"""
    )

    return {"destination": destination, "flights": flights, "hotels": hotels,
            "events": events, "restaurants": restaurants, "summary": summary}

trip = await travel_plan("Tokyo, Japan", "April 10-17 2025", "mid-range")

print(f"\n🗺️  TRAVEL PLAN SUMMARY:\n{trip['summary']}")
print(f"\n✈️  Flights:\n{trip['flights']}")
print(f"\n🏨 Hotels:\n{trip['hotels']}")

## 10. Use Case 4 — A/B Content Generation

Generate multiple variations of content simultaneously, then select the best.

In [ ]:
PRODUCT = "An AI-powered project management tool for remote teams"

async def generate_headline(product: str, style: str) -> str:
    return await acall_llm(
        f"Write one {style} marketing headline for: {product}. One sentence only.",
        system="You are an expert copywriter."
    )

async def ab_generate(product: str) -> dict:
    styles = ["bold and punchy", "professional and trustworthy", "friendly and conversational"]

    print(f"🎨 Generating {len(styles)} headline variants in parallel...")
    t0 = time.perf_counter()

    headlines = await asyncio.gather(*[generate_headline(product, s) for s in styles])

    elapsed = time.perf_counter() - t0
    print(f"✅ {len(styles)} variants generated in {elapsed:.2f}s")

    variants = []
    for style, headline in zip(styles, headlines):
        variants.append({"style": style, "headline": headline})
        print(f"\n  [{style.upper()}]\n  {headline}")

    options_text = "\n".join(f"{i+1}. {h}" for i, h in enumerate(headlines))
    best = await acall_llm(
        f"""Which of these headlines is most effective for '{product}'?
Reply with the number only (1, 2, or 3).

{options_text}"""
    )

    best_idx = int(best.strip()[0]) - 1
    return {"variants": variants, "winner_index": best_idx,
            "winner": variants[best_idx]["headline"]}

ab_result = await ab_generate(PRODUCT)
print(f"\n🏆 Winning Headline:\n   {ab_result['winner']}")

## 11. Use Case 5 — Parallel Input Validation

Run multiple validation checks simultaneously on user-submitted data.

In [ ]:
import re

def check_email(email: str) -> dict:
    valid = bool(re.match(r'^[\w.+-]+@[\w-]+\.[\w.-]+$', email))
    return {"check": "email_format", "passed": valid, "message": "Valid" if valid else "Invalid email format"}

def check_phone(phone: str) -> dict:
    valid = bool(re.match(r'^\+?[1-9]\d{6,14}$', phone.replace(" ", "").replace("-", "")))
    return {"check": "phone_format", "passed": valid, "message": "Valid" if valid else "Invalid phone number"}

def check_age(age_str: str) -> dict:
    try:
        age = int(age_str)
        valid = 18 <= age <= 120
        return {"check": "age_range", "passed": valid, "message": f"Age {age} valid" if valid else f"Age {age} out of range"}
    except ValueError:
        return {"check": "age_range", "passed": False, "message": "Age must be a number"}

async def check_profanity(text: str) -> dict:
    result = await acall_llm(
        f"Does this text contain profanity or inappropriate content? Reply YES or NO only: {text}"
    )
    clean = result.strip().upper().startswith("NO")
    return {"check": "content_policy", "passed": clean, "message": "Clean" if clean else "Contains inappropriate content"}

async def validate_user_input(email: str, phone: str, age: str, username: str) -> dict:
    print(f"🔍 Running 4 validation checks in parallel...")
    t0 = time.perf_counter()

    loop = asyncio.get_event_loop()
    email_check, phone_check, age_check, profanity_check = await asyncio.gather(
        loop.run_in_executor(None, check_email, email),
        loop.run_in_executor(None, check_phone, phone),
        loop.run_in_executor(None, check_age, age),
        check_profanity(username)
    )

    elapsed = time.perf_counter() - t0
    checks = [email_check, phone_check, age_check, profanity_check]
    all_passed = all(c["passed"] for c in checks)

    print(f"✅ Validation complete in {elapsed:.2f}s")
    return {"all_passed": all_passed, "checks": checks}

print("=== VALID USER ===")
result = await validate_user_input("phunkie@wragby.com", "+2348012345678", "32", "phunkie")
print(f"Overall: {'✅ PASS' if result['all_passed'] else '❌ FAIL'}")
for c in result["checks"]:
    icon = "✅" if c["passed"] else "❌"
    print(f"  {icon} {c['check']}: {c['message']}")

print("\n=== INVALID USER ===")
result2 = await validate_user_input("not-an-email", "123", "15", "badword_user")
print(f"Overall: {'✅ PASS' if result2['all_passed'] else '❌ FAIL'}")
for c in result2["checks"]:
    icon = "✅" if c["passed"] else "❌"
    print(f"  {icon} {c['check']}: {c['message']}")

## 12. Use Case 6 — Parallel Document Processing (Chunk-Level)

Split a long document into chunks, process each chunk in parallel, then merge.

In [ ]:
LONG_DOCUMENT = """
Chapter 1: The renewable energy sector saw unprecedented growth in 2024, with solar 
installations increasing by 45% globally. Key markets included China, India, and the 
United States. Battery storage capacity also tripled compared to 2022 levels.

Chapter 2: Electric vehicle adoption continued its rapid pace, with EV sales reaching 
18 million units worldwide. Charging infrastructure expanded to over 3 million public 
stations globally. Range anxiety decreased significantly as average EV range surpassed 
400 km per charge.

Chapter 3: Carbon capture technology achieved a major milestone with the first 
commercial-scale direct air capture plant reaching 1 million tonnes per year capacity. 
Government subsidies totalled $4.2 billion across OECD nations. Critics argue costs 
remain prohibitively high at $300 per tonne.
"""

def chunk_document(text: str, chunk_size: int = 300) -> list:
    'Split document into roughly equal chunks by paragraph.'
    paragraphs = [p.strip() for p in text.strip().split("\n\n") if p.strip()]
    return paragraphs

async def process_chunk(chunk: str, chunk_id: int) -> dict:
    'Extract key facts from a single chunk.'
    facts = await acall_llm(
        f"Extract 2 key facts as bullet points from this text:\n{chunk}",
        system="You are a precise fact extractor."
    )
    return {"chunk_id": chunk_id, "text_preview": chunk[:60], "facts": facts}

async def parallel_document_processor(document: str) -> str:
    chunks = chunk_document(document)
    print(f"📄 Document split into {len(chunks)} chunks — processing in parallel...")
    t0 = time.perf_counter()

    chunk_results = await asyncio.gather(
        *[process_chunk(chunk, i) for i, chunk in enumerate(chunks)]
    )

    elapsed = time.perf_counter() - t0
    print(f"✅ All chunks processed in {elapsed:.2f}s")

    all_facts = []
    for r in sorted(chunk_results, key=lambda x: x["chunk_id"]):
        print(f"\n[Chunk {r['chunk_id']}] ...{r['text_preview']}...")
        print(r["facts"])
        all_facts.append(r["facts"])

    merged_facts = "\n".join(all_facts)
    final_summary = await acall_llm(
        f"Synthesise these facts into a 3-sentence executive summary:\n{merged_facts}",
        system="You are an executive report writer."
    )
    return final_summary

executive_summary = await parallel_document_processor(LONG_DOCUMENT)
print(f"\n📋 EXECUTIVE SUMMARY:\n{executive_summary}")

## 13. Benchmark — Sequential vs Parallel (Side-by-Side)

In [ ]:
BENCHMARK_TOPICS = [
    "Machine learning",
    "Climate change",
    "Blockchain technology",
    "Gene editing",
]

async def benchmark():
    print("=" * 55)
    print("BENCHMARK: Sequential vs Parallel LLM calls")
    print(f"Tasks: {len(BENCHMARK_TOPICS)} summarisation calls")
    print("=" * 55)

    print("\n[1/2] Running SEQUENTIALLY...")
    t0 = time.perf_counter()
    for topic in BENCHMARK_TOPICS:
        await acall_llm(f"Summarise in 1 sentence: {topic}")
    seq_t = time.perf_counter() - t0

    print("[2/2] Running IN PARALLEL...")
    t0 = time.perf_counter()
    await asyncio.gather(
        *[acall_llm(f"Summarise in 1 sentence: {t}") for t in BENCHMARK_TOPICS]
    )
    par_t = time.perf_counter() - t0

    speedup = seq_t / par_t
    print(f"""
╔══════════════════════════════════╗
║ BENCHMARK RESULTS                ║
╠══════════════════════════════════╣
║ Sequential : {seq_t:>6.2f}s             ║
║ Parallel   : {par_t:>6.2f}s             ║
║ Speedup    : {speedup:>6.1f}x faster       ║
║ Time saved : {seq_t-par_t:>6.2f}s             ║
╚══════════════════════════════════╝
    """)
    return seq_t, par_t, speedup

seq_t, par_t, speedup = await benchmark()

## 14. Error Handling in Parallel Workflows

When one parallel task fails, we shouldn't lose all results. Use `asyncio.gather(return_exceptions=True)` or `asyncio.wait`.

In [ ]:
async def unreliable_task(name: str, fail: bool = False) -> str:
    'Simulates a task that may fail.'
    if fail:
        raise ConnectionError(f"Task '{name}' failed: API timeout")
    return await acall_llm(f"Write one interesting fact about {name}.")

async def robust_parallel_run(tasks_config: list) -> list:
    'Run tasks in parallel with graceful error handling.'
    tasks = [unreliable_task(name, fail) for name, fail in tasks_config]

    results = await asyncio.gather(*tasks, return_exceptions=True)

    output = []
    for (name, _), result in zip(tasks_config, results):
        if isinstance(result, Exception):
            output.append({"name": name, "status": "FAILED",  "data": str(result)})
        else:
            output.append({"name": name, "status": "SUCCESS", "data": result})
    return output

tasks_config = [
    ("Python",       False),
    ("JavaScript",   True),
    ("Rust",         False),
    ("Go",           True),
]

print("=== ROBUST PARALLEL EXECUTION WITH ERROR HANDLING ===")
results = await robust_parallel_run(tasks_config)
for r in results:
    icon = "✅" if r["status"] == "SUCCESS" else "❌"
    print(f"\n{icon} [{r['status']}] {r['name']}")
    print(f"   {r['data'][:120]}")

successful = [r for r in results if r["status"] == "SUCCESS"]
print(f"\n📊 {len(successful)}/{len(results)} tasks completed successfully")

## 15. Summary — Pattern Rules of Thumb

In [ ]:
summary = """
╔═══════════════════════════════════╦════════════════════════════════════════════╗
║ When to USE Parallelization       ║ Key Design Rules                           ║
╠═══════════════════════════════════╬════════════════════════════════════════════╣
║ Tasks are independent of each    ║ Always have a sequential synthesis step    ║
║ other (no data dependency)        ║ after parallel tasks converge             ║
║ Tasks involve I/O wait            ║ Use asyncio.gather for async APIs          ║
║ (API calls, DB queries)           ║ Use ThreadPoolExecutor for sync APIs       ║
║ You need results from multiple   ║ Handle errors with return_exceptions=True  ║
║ sources before synthesising       ║ Set max_workers/concurrency limits         ║
║ Batch processing is required      ║ Log and monitor parallel task outcomes     ║
╚═══════════════════════════════════╩════════════════════════════════════════════╝

⚠️  Complexity Warning:
   Parallel architectures are harder to debug than sequential ones.
   Add structured logging to every parallel task in production.
"""
print(summary)

## 16. Exercises

- **Exercise 1 — Rate Limiting:** Modify `run_all_async` to limit concurrency to a maximum of 2 simultaneous calls using `asyncio.Semaphore`.

- **Exercise 2 — Retry Logic:** Add exponential backoff retry (max 3 attempts) to `unreliable_task` before marking it as failed.

- **Exercise 3 — Progress Bar:** Add a `tqdm.asyncio` progress bar to `parallel_document_processor` (install `pip install tqdm`) that updates as each chunk completes.

- **Exercise 4 — Parallel + Routing:** Combine Chapter 2 (routing) and Chapter 3 (parallelization): build a system that routes an incoming request to the correct domain, then fans out 3 parallel sub-tasks within that domain.

- **Exercise 5 — ProcessPoolExecutor:** Modify the document processor (Section 12) to use `concurrent.futures.ProcessPoolExecutor` instead of asyncio. Benchmark the difference.

**Next:** Chapter 4 — Orchestrator-Worker Pattern